# Product Sales and Profit Analysis Using Python and Excel
### B.Sc. Data Science Mini Project | Exploratory Data Analysis & Financial Modeling
---
**Author / Student:** [STUDENT NAME]  
**Roll Number:** [ROLL NUMBER]  
**Institution:** [COLLEGE / INSTITUTION NAME]  
**Academic Year:** 2024 - 2025  

#### Project Objective:
To develop an end-to-end data analysis solution using Python and Microsoft Excel to evaluate sales performance, revenue, total cost of goods sold, net profit, operating profit margin, and multi-dimensional segment trends (product-wise, category-wise, regional, and monthly).

## 1. Import Required Libraries
We import `pandas` and `numpy` for data manipulation, `matplotlib.pyplot` and `seaborn` for graphical visualizations, and `openpyxl` for spreadsheet management.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic parameters
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries successfully imported!")

## 2. Load the Dataset & Initial Exploration
The dataset contains transactional sales logs across 2024. Let us inspect the raw schema, column data types, and head rows.

In [2]:
# Load the raw CSV dataset
csv_path = os.path.join("..", "data", "sales_data.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join("data", "sales_data.csv")

raw_df = pd.read_csv(csv_path)
print(f"Raw dataset shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns\n")
raw_df.head(10)

In [3]:
# Inspect metadata and summary statistics
raw_df.info()
raw_df.describe()

## 3. Data Cleaning and Quality Validation
Real-world data often suffers from missing values, duplicate transactions, and out-of-boundary numbers. Let us inspect and clean these anomalies systematically.

In [4]:
# Check for missing (null) values
print("--- Missing Values Count ---")
print(raw_df.isnull().sum())

# Check for duplicate records
num_duplicates = raw_df.duplicated().sum()
print(f"\nExact duplicate rows detected: {num_duplicates}")

# Check for non-positive quantities or prices
invalid_qty = (raw_df["Quantity"] <= 0).sum()
invalid_price = ((raw_df["Selling_Price"] <= 0) | (raw_df["Cost_Price"] <= 0)).sum()
print(f"Invalid Quantity rows (<= 0): {invalid_qty}")
print(f"Invalid Price rows (<= 0): {invalid_price}")

In [5]:
# Execute cleaning transformations
clean_df = raw_df.copy()

# 1. Drop duplicate records
clean_df = clean_df.drop_duplicates()

# 2. Drop rows missing critical identifiers or numerical values
clean_df = clean_df.dropna(subset=["Order_ID", "Order_Date", "Product_ID", "Product_Name", "Category", "Region", "Quantity", "Selling_Price", "Cost_Price"])

# 3. Enforce strictly positive quantities and prices
clean_df = clean_df[(clean_df["Quantity"] > 0) & (clean_df["Selling_Price"] > 0) & (clean_df["Cost_Price"] > 0)]

# 4. Parse Order_Date to datetime and sort chronologically
clean_df["Order_Date"] = pd.to_datetime(clean_df["Order_Date"])
clean_df["Quantity"] = clean_df["Quantity"].astype(int)
clean_df = clean_df.sort_values("Order_Date").reset_index(drop=True)

print(f"Cleaning Summary: {len(raw_df)} initial rows -> {len(clean_df)} validated rows retained.")
print(f"Total invalid/duplicate records removed: {len(raw_df) - len(clean_df)}")

## 4. Financial Calculations
We calculate standard commercial metrics:
- $\text{Revenue} = \text{Quantity} \times \text{Selling\_Price}$
- $\text{Total\_Cost} = \text{Quantity} \times \text{Cost\_Price}$
- $\text{Profit} = \text{Revenue} - \text{Total\_Cost}$
- $\text{Profit\_Margin (\%)} = \left(\frac{\text{Profit}}{\text{Revenue}}\right) \times 100$

In [6]:
# Vectorized financial calculations
clean_df["Revenue"] = (clean_df["Quantity"] * clean_df["Selling_Price"]).round(2)
clean_df["Total_Cost"] = (clean_df["Quantity"] * clean_df["Cost_Price"]).round(2)
clean_df["Profit"] = (clean_df["Revenue"] - clean_df["Total_Cost"]).round(2)

# Handle division by zero safely
clean_df["Profit_Margin"] = np.where(
    clean_df["Revenue"] > 0,
    ((clean_df["Profit"] / clean_df["Revenue"]) * 100.0).round(2),
    0.0
)

# Temporal feature extraction
clean_df["Year_Month"] = clean_df["Order_Date"].dt.strftime("%Y-%m")
clean_df["Month_Name"] = clean_df["Order_Date"].dt.strftime("%b")

clean_df[["Order_ID", "Product_Name", "Quantity", "Selling_Price", "Revenue", "Total_Cost", "Profit", "Profit_Margin"]].head()

## 5. Executive Key Performance Indicators (KPIs)
Let us summarize high-level business performance metrics across the entire year 2024.

In [7]:
total_revenue = clean_df["Revenue"].sum()
total_cost = clean_df["Total_Cost"].sum()
total_profit = clean_df["Profit"].sum()
overall_margin = (total_profit / total_revenue) * 100
total_orders = clean_df["Order_ID"].nunique()
total_quantity = clean_df["Quantity"].sum()
avg_order_val = clean_df["Revenue"].mean()

kpi_df = pd.DataFrame({
    "Metric": [
        "Total Sales Revenue",
        "Total Cost of Goods Sold",
        "Net Operating Profit",
        "Overall Profit Margin",
        "Total Orders Processed",
        "Total Quantity Sold",
        "Average Order Value (AOV)"
    ],
    "Value": [
        f"INR {total_revenue:,.2f}",
        f"INR {total_cost:,.2f}",
        f"INR {total_profit:,.2f}",
        f"{overall_margin:.2f}%\掠,
        f"{total_orders:,}",
        f"{total_quantity:,} units",
        f"INR {avg_order_val:,.2f}"
    ]
})
kpi_df

## 6. Multi-Dimensional Performance Analysis
We group data across products, categories, geographical regions, and calendar months.

In [8]:
# 1. Product-Wise Performance Summary
prod_summary = clean_df.groupby(["Product_ID", "Product_Name", "Category"]).agg(
    Orders=("Order_ID", "count"),
    Units_Sold=("Quantity", "sum"),
    Revenue=("Revenue", "sum"),
    Total_Cost=("Total_Cost", "sum"),
    Profit=("Profit", "sum")
).reset_index()
prod_summary["Profit_Margin"] = (prod_summary["Profit"] / prod_summary["Revenue"] * 100).round(2)
prod_summary = prod_summary.sort_values("Profit", ascending=False).reset_index(drop=True)
prod_summary

In [9]:
# 2. Category-Wise Performance
cat_summary = clean_df.groupby("Category").agg(
    Orders=("Order_ID", "count"),
    Units_Sold=("Quantity", "sum"),
    Revenue=("Revenue", "sum"),
    Total_Cost=("Total_Cost", "sum"),
    Profit=("Profit", "sum")
).reset_index()
cat_summary["Profit_Margin"] = (cat_summary["Profit"] / cat_summary["Revenue"] * 100).round(2)
cat_summary["Revenue_Share_Pct"] = (cat_summary["Revenue"] / total_revenue * 100).round(2)
cat_summary = cat_summary.sort_values("Revenue", ascending=False).reset_index(drop=True)
cat_summary

In [10]:
# 3. Regional Performance
reg_summary = clean_df.groupby("Region").agg(
    Orders=("Order_ID", "count"),
    Units_Sold=("Quantity", "sum"),
    Revenue=("Revenue", "sum"),
    Profit=("Profit", "sum")
).reset_index()
reg_summary["Profit_Margin"] = (reg_summary["Profit"] / reg_summary["Revenue"] * 100).round(2)
reg_summary = reg_summary.sort_values("Revenue", ascending=False).reset_index(drop=True)
reg_summary

In [11]:
# 4. Monthly Progression
mon_summary = clean_df.groupby(["Year_Month", "Month_Name"]).agg(
    Orders=("Order_ID", "count"),
    Units_Sold=("Quantity", "sum"),
    Revenue=("Revenue", "sum"),
    Total_Cost=("Total_Cost", "sum"),
    Profit=("Profit", "sum")
).reset_index().sort_values("Year_Month").reset_index(drop=True)
mon_summary["Profit_Margin"] = (mon_summary["Profit"] / mon_summary["Revenue"] * 100).round(2)
mon_summary

## 7. Strategic Business Insights
- **Best-Selling Product by Volume:** Identifying which SKU moved the largest number of physical units.
- **Best-Selling Product by Revenue:** The SKU contributing the highest top-line turnover.
- **Most Profitable Product:** The product generating maximum absolute operating profit.
- **Lowest-Performing Product:** The product delivering lowest absolute profit (enabling review of inventory or pricing).

In [12]:
best_vol = prod_summary.sort_values("Units_Sold", ascending=False).iloc[0]
best_rev = prod_summary.sort_values("Revenue", ascending=False).iloc[0]
most_prof = prod_summary.sort_values("Profit", ascending=False).iloc[0]
lowest_prof = prod_summary.sort_values("Profit", ascending=True).iloc[0]

print(f"[*] Best-Selling Product (Units):   {best_vol['Product_Name']} ({best_vol['Units_Sold']} units)")
print(f"[*] Best-Selling Product (Revenue): {best_rev['Product_Name']} (INR {best_rev['Revenue']:,.2f})")
print(f"[*] Most Profitable Product:        {most_prof['Product_Name']} (INR {most_prof['Profit']:,.2f})")
print(f"[*] Lowest-Performing Product:      {lowest_prof['Product_Name']} (INR {lowest_prof['Profit']:,.2f})")

## 8. Data Visualizations
We plot publication-quality charts using Matplotlib.

In [13]:
# Chart 1: Product-Wise Net Profit
plt.figure(figsize=(10, 6))
sorted_p = prod_summary.sort_values("Profit", ascending=True)
bars = plt.barh(sorted_p["Product_Name"], sorted_p["Profit"] / 1000, color="#2b5c8f", edgecolor="black")
plt.title("Product-Wise Net Profit (INR in Thousands)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Net Profit (Thousands INR)", fontsize=11, fontweight="bold")
plt.ylabel("Product Name", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

In [14]:
# Chart 2: Monthly Trends in Revenue, Cost & Profit
plt.figure(figsize=(11, 5.5))
plt.plot(mon_summary["Month_Name"], mon_summary["Revenue"] / 1000, marker="o", linewidth=2.5, color="#0d6efd", label="Revenue")
plt.plot(mon_summary["Month_Name"], mon_summary["Total_Cost"] / 1000, marker="s", linewidth=2, linestyle="--", color="#dc3545", label="Total Cost")
plt.plot(mon_summary["Month_Name"], mon_summary["Profit"] / 1000, marker="^", linewidth=2.5, color="#198754", label="Net Profit")
plt.fill_between(mon_summary["Month_Name"], mon_summary["Profit"] / 1000, color="#198754", alpha=0.15)
plt.title("Monthly Sales, Cost & Profit Trends (2024)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Month", fontsize=11, fontweight="bold")
plt.ylabel("Amount (INR in Thousands)", fontsize=11, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

In [15]:
# Chart 3: Category Revenue & Profit Breakdown
x = np.arange(len(cat_summary))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, cat_summary["Revenue"] / 100000, width, label="Revenue", color="#1f77b4")
plt.bar(x + width/2, cat_summary["Profit"] / 100000, width, label="Net Profit", color="#2ca02c")
plt.xticks(x, cat_summary["Category"], fontweight="bold")
plt.title("Category-Wise Performance: Revenue vs Profit (INR in Lakhs)", fontsize=13, fontweight="bold", pad=12)
plt.ylabel("Amount (INR in Lakhs)", fontsize=11, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Conclusion
This comprehensive analysis illustrates how combining Python's high-speed programmatic wrangling with Microsoft Excel's accessible dashboarding provides a scalable, auditable business intelligence framework for commercial enterprise management.